# Mathematical Fundamentals for Machine Learning
## Single-Variable Regression with Gradient Descent

In this notebook, we'll build up the mathematical tools needed to understand how a machine learning model learns. We'll start with lines, then cost functions, then derivatives, and finally see how gradient descent uses derivatives to find the best-fitting line.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

# Set random seed for reproducibility
np.random.seed(42)

## Part 1: The Equation of a Line

A line is a function that maps an input x to an output y.

$$y = mx + b$$

- **m** is the slope: how steep the line is
- **b** is the intercept: where the line crosses the y-axis

In [ ]:
# Let's visualize how changing m and b affects the line
x = np.linspace(-5, 5, 100)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left plot: vary slope m
ax = axes[0]
slopes = [0.5, 1, 2, -1]
colors = ['blue', 'orange', 'green', 'red']
for m, color in zip(slopes, colors):
    y = m * x + 0  # b = 0
    ax.plot(x, y, label=f'm = {m}', color=color, linewidth=2)
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_title('Effect of slope (m) on the line')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_ylim([-10, 10])

# Right plot: vary intercept b
ax = axes[1]
intercepts = [-2, 0, 2, 4]
for b, color in zip(intercepts, colors):
    y = 1 * x + b  # m = 1
    ax.plot(x, y, label=f'b = {b}', color=color, linewidth=2)
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_title('Effect of intercept (b) on the line')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_ylim([-10, 10])

plt.tight_layout()
plt.show()

print("Key insight: m and b are parameters we can adjust to change the line.")
print("In machine learning, we'll find the m and b that best fit our data.")

## Part 2: The Cost Function (Mean Squared Error)

Given some data, we want to find the line that fits it best. But what does "best" mean?

We define a **cost function** that measures how badly a line fits the data. We'll use **Mean Squared Error (MSE)**:

$$\text{Cost}(m, b) = \frac{1}{n} \sum_{i=1}^{n} (y_i - (mx_i + b))^2$$

Where:
- $(x_i, y_i)$ are our data points
- $(mx_i + b)$ is our prediction for $x_i$
- We square the errors to penalize large mistakes and avoid negatives canceling positives
- A lower cost means a better fit

In [ ]:
# Generate some sample data
n_points = 30
x_data = np.linspace(0, 10, n_points)
# True relationship: y ≈ 2x + 3, plus some noise
y_data = 2 * x_data + 3 + np.random.normal(0, 3, n_points)

# Define the cost function
def mse_cost(x, y, m, b):
    """Calculate mean squared error for a given line (m, b)"""
    predictions = m * x + b
    errors = y - predictions
    return np.mean(errors ** 2)

# Visualize the data and a few candidate lines
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left plot: data with different candidate lines
ax = axes[0]
ax.scatter(x_data, y_data, s=50, alpha=0.6, label='Data', color='black')

candidates = [(1, 0), (2, 3), (1.5, 5), (0.5, 8)]
colors = ['blue', 'green', 'orange', 'red']
for (m, b), color in zip(candidates, colors):
    y_pred = m * x_data + b
    cost = mse_cost(x_data, y_data, m, b)
    ax.plot(x_data, y_pred, label=f'm={m}, b={b}, cost={cost:.1f}', color=color, linewidth=2, alpha=0.7)

ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_title('Different candidate lines and their costs')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# Right plot: cost surface as a function of m and b
ax = axes[1]
m_range = np.linspace(-2, 5, 50)
b_range = np.linspace(-5, 15, 50)
M, B = np.meshgrid(m_range, b_range)
Z = np.zeros_like(M)
for i in range(M.shape[0]):
    for j in range(M.shape[1]):
        Z[i, j] = mse_cost(x_data, y_data, M[i, j], B[i, j])

contour = ax.contourf(M, B, Z, levels=20, cmap='viridis')
ax.set_xlabel('m (slope)')
ax.set_ylabel('b (intercept)')
ax.set_title('Cost function as a function of m and b')
plt.colorbar(contour, ax=ax, label='Cost')

plt.tight_layout()
plt.show()

print("Key insight: The cost function tells us how good any choice of m and b is.")
print("Lower cost = better fit. Our goal is to find m and b that minimize the cost.")

## Part 3: Derivatives

A **derivative** measures how much a function changes when you nudge its input slightly.

$$\frac{df}{dx} = \lim_{h \to 0} \frac{f(x+h) - f(x)}{h}$$

Intuitively: it's the **slope** of the function at that point.

### Rules for computing derivatives

**Power rule:** If $f(x) = x^n$, then $\frac{df}{dx} = nx^{n-1}$

**Sum rule:** If $f(x) = g(x) + h(x)$, then $\frac{df}{dx} = \frac{dg}{dx} + \frac{dh}{dx}$

**Constant multiple:** If $f(x) = c \cdot g(x)$, then $\frac{df}{dx} = c \cdot \frac{dg}{dx}$

In [ ]:
# Let's visualize derivatives as slopes
x_vals = np.linspace(-3, 3, 100)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Example 1: f(x) = x^2
ax = axes[0]
f1 = lambda x: x**2
df1 = lambda x: 2*x  # derivative

ax.plot(x_vals, f1(x_vals), 'b-', linewidth=2, label='f(x) = x²')

# Mark some points and draw tangent lines
points = [-2, 0, 1.5]
for x_pt in points:
    y_pt = f1(x_pt)
    slope = df1(x_pt)
    # Draw tangent line
    x_tangent = np.linspace(x_pt - 1, x_pt + 1, 20)
    y_tangent = y_pt + slope * (x_tangent - x_pt)
    ax.plot(x_tangent, y_tangent, 'r--', alpha=0.5, linewidth=1)
    ax.plot(x_pt, y_pt, 'ro', markersize=6)
    ax.text(x_pt, y_pt + 0.5, f"f'={slope:.1f}", fontsize=9, ha='center')

ax.set_xlabel('x')
ax.set_ylabel('f(x)')
ax.set_title('f(x) = x² and its tangent lines (slopes = derivatives)')
ax.grid(True, alpha=0.3)
ax.legend()
ax.set_ylim([-1, 9])

# Example 2: f(x) = x³ - 3x
ax = axes[1]
f2 = lambda x: x**3 - 3*x
df2 = lambda x: 3*x**2 - 3  # derivative

ax.plot(x_vals, f2(x_vals), 'b-', linewidth=2, label='f(x) = x³ - 3x')
ax.plot(x_vals, df2(x_vals), 'g-', linewidth=2, label="f'(x) = 3x² - 3")

# Mark where derivative = 0 (critical points)
critical_points = [-1, 1]
for x_pt in critical_points:
    ax.plot(x_pt, f2(x_pt), 'ro', markersize=8)
    ax.text(x_pt, f2(x_pt) - 1, 'min/max', fontsize=9, ha='center')

ax.set_xlabel('x')
ax.set_ylabel('f(x) or f\'(x)')
ax.set_title('Function and its derivative')
ax.grid(True, alpha=0.3)
ax.legend()
ax.axhline(y=0, color='k', linestyle='-', linewidth=0.5)

# Example 3: Cost function and its derivative (with respect to m)
ax = axes[2]
m_vals = np.linspace(-1, 4, 100)
costs = [mse_cost(x_data, y_data, m, 3) for m in m_vals]

ax.plot(m_vals, costs, 'b-', linewidth=2, label='Cost(m)')

# Approximate derivative numerically
dcosts = np.gradient(costs, m_vals)
ax.plot(m_vals, dcosts, 'g-', linewidth=2, label="Cost'(m)")
ax.axhline(y=0, color='k', linestyle='-', linewidth=0.5)

ax.set_xlabel('m (slope)')
ax.set_ylabel('Cost or Cost\'(m)')
ax.set_title('Cost function and its derivative (b=3 fixed)')
ax.grid(True, alpha=0.3)
ax.legend()

plt.tight_layout()
plt.show()

print("Key insights:")
print("1. The derivative is the slope at a point.")
print("2. Where the derivative = 0, the function reaches a minimum or maximum.")
print("3. For our cost function, where Cost'(m) = 0 is where we find the best m.")

## Part 4: Finding the Derivative of the Cost Function

Our cost function is:
$$\text{Cost}(m, b) = \frac{1}{n} \sum_{i=1}^{n} (y_i - (mx_i + b))^2$$

To minimize it with respect to $m$, we take the derivative and set it to zero:
$$\frac{\partial \text{Cost}}{\partial m} = \frac{1}{n} \sum_{i=1}^{n} 2(y_i - (mx_i + b)) \cdot (-x_i)$$

A **partial derivative** tells us how the cost changes when we nudge just one parameter (m or b), keeping others fixed.

This formula tells us: if the derivative is negative, increasing m will decrease cost. If positive, decreasing m will decrease cost.

In [ ]:
# Define the derivative of cost with respect to m
def cost_derivative_m(x, y, m, b):
    """Compute ∂Cost/∂m"""
    predictions = m * x + b
    errors = y - predictions
    return -2/len(x) * np.sum(errors * x)

# Define the derivative of cost with respect to b
def cost_derivative_b(x, y, m, b):
    """Compute ∂Cost/∂b"""
    predictions = m * x + b
    errors = y - predictions
    return -2/len(x) * np.sum(errors)

# Visualize the derivatives of the cost function
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Derivative with respect to m
ax = axes[0]
m_range = np.linspace(-1, 4, 50)
derivative_m = [cost_derivative_m(x_data, y_data, m, 3) for m in m_range]
costs_m = [mse_cost(x_data, y_data, m, 3) for m in m_range]

ax_twin = ax.twinx()
ax.plot(m_range, costs_m, 'b-', linewidth=2, label='Cost(m)')
ax_twin.plot(m_range, derivative_m, 'r-', linewidth=2, label="∂Cost/∂m")
ax_twin.axhline(y=0, color='k', linestyle='--', linewidth=1, alpha=0.5)

ax.set_xlabel('m (slope)')
ax.set_ylabel('Cost(m)', color='b')
ax_twin.set_ylabel('∂Cost/∂m', color='r')
ax.set_title('Cost and its derivative with respect to m (b=3)')
ax.grid(True, alpha=0.3)
ax.tick_params(axis='y', labelcolor='b')
ax_twin.tick_params(axis='y', labelcolor='r')

# Derivative with respect to b
ax = axes[1]
b_range = np.linspace(-5, 15, 50)
derivative_b = [cost_derivative_b(x_data, y_data, 2, b) for b in b_range]
costs_b = [mse_cost(x_data, y_data, 2, b) for b in b_range]

ax_twin = ax.twinx()
ax.plot(b_range, costs_b, 'b-', linewidth=2, label='Cost(b)')
ax_twin.plot(b_range, derivative_b, 'r-', linewidth=2, label="∂Cost/∂b")
ax_twin.axhline(y=0, color='k', linestyle='--', linewidth=1, alpha=0.5)

ax.set_xlabel('b (intercept)')
ax.set_ylabel('Cost(b)', color='b')
ax_twin.set_ylabel('∂Cost/∂b', color='r')
ax.set_title('Cost and its derivative with respect to b (m=2)')
ax.grid(True, alpha=0.3)
ax.tick_params(axis='y', labelcolor='b')
ax_twin.tick_params(axis='y', labelcolor='r')

plt.tight_layout()
plt.show()

print("Key insight: The derivative tells us which direction to nudge our parameters.")
print("- If ∂Cost/∂m is positive, we should decrease m")
print("- If ∂Cost/∂m is negative, we should increase m")
print("- When the derivative = 0, we've found the optimal value!")

## Part 5: Gradient Descent

Instead of solving for where the derivative equals zero analytically, we can use an iterative algorithm called **gradient descent**.

The idea: 
1. Start with a guess for m and b
2. Compute the derivatives ∂Cost/∂m and ∂Cost/∂b
3. Take a small step in the direction opposite to the gradient
4. Repeat until the cost stops decreasing

The update rule is:
$$m_{\text{new}} = m_{\text{old}} - \alpha \cdot \frac{\partial \text{Cost}}{\partial m}$$
$$b_{\text{new}} = b_{\text{old}} - \alpha \cdot \frac{\partial \text{Cost}}{\partial b}$$

Where $\alpha$ (alpha) is the **learning rate**: how big a step we take. Too small = slow learning. Too big = might overshoot.

In [ ]:
# Implement gradient descent
def gradient_descent(x, y, m_init, b_init, learning_rate, num_iterations):
    """
    Perform gradient descent to find optimal m and b.
    
    Returns:
    - m_history, b_history, cost_history: values at each iteration
    """
    m, b = m_init, b_init
    m_history, b_history, cost_history = [m], [b], [mse_cost(x, y, m, b)]
    
    for iteration in range(num_iterations):
        # Compute gradients
        dm = cost_derivative_m(x, y, m, b)
        db = cost_derivative_b(x, y, m, b)
        
        # Update parameters
        m = m - learning_rate * dm
        b = b - learning_rate * db
        
        # Track history
        m_history.append(m)
        b_history.append(b)
        cost_history.append(mse_cost(x, y, m, b))
    
    return m_history, b_history, cost_history

# Run gradient descent
m_init, b_init = 0.5, 5
learning_rate = 0.01
num_iterations = 100

m_hist, b_hist, cost_hist = gradient_descent(x_data, y_data, m_init, b_init, learning_rate, num_iterations)

print(f"Initial: m={m_init:.3f}, b={b_init:.3f}, cost={cost_hist[0]:.3f}")
print(f"Final:   m={m_hist[-1]:.3f}, b={b_hist[-1]:.3f}, cost={cost_hist[-1]:.3f}")
print(f"\nAfter {num_iterations} iterations, we've found a good fit!")

In [ ]:
# Visualize gradient descent
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Top-left: Cost function landscape with gradient descent path
ax = axes[0, 0]
m_range = np.linspace(-1, 4, 50)
b_range = np.linspace(-5, 15, 50)
M, B = np.meshgrid(m_range, b_range)
Z = np.zeros_like(M)
for i in range(M.shape[0]):
    for j in range(M.shape[1]):
        Z[i, j] = mse_cost(x_data, y_data, M[i, j], B[i, j])

contour = ax.contourf(M, B, Z, levels=15, cmap='viridis', alpha=0.8)
ax.plot(m_hist, b_hist, 'r.-', linewidth=2, markersize=4, label='Gradient descent path')
ax.plot(m_hist[0], b_hist[0], 'go', markersize=10, label='Start')
ax.plot(m_hist[-1], b_hist[-1], 'r*', markersize=15, label='End')
ax.set_xlabel('m (slope)')
ax.set_ylabel('b (intercept)')
ax.set_title('Gradient descent path on cost landscape')
ax.legend()
plt.colorbar(contour, ax=ax, label='Cost')

# Top-right: Cost decreasing over iterations
ax = axes[0, 1]
ax.plot(range(len(cost_hist)), cost_hist, 'b-', linewidth=2)
ax.set_xlabel('Iteration')
ax.set_ylabel('Cost')
ax.set_title('Cost decreases with each iteration')
ax.grid(True, alpha=0.3)

# Bottom-left: m and b converging
ax = axes[1, 0]
ax.plot(range(len(m_hist)), m_hist, 'b-', linewidth=2, label='m (slope)')
ax.plot(range(len(b_hist)), b_hist, 'r-', linewidth=2, label='b (intercept)')
ax.axhline(y=m_hist[-1], color='b', linestyle='--', alpha=0.5)
ax.axhline(y=b_hist[-1], color='r', linestyle='--', alpha=0.5)
ax.set_xlabel('Iteration')
ax.set_ylabel('Parameter value')
ax.set_title('Parameters converge to optimal values')
ax.legend()
ax.grid(True, alpha=0.3)

# Bottom-right: Final fitted line
ax = axes[1, 1]
ax.scatter(x_data, y_data, s=50, alpha=0.6, label='Data', color='black')
x_line = np.linspace(x_data.min(), x_data.max(), 100)
y_line = m_hist[-1] * x_line + b_hist[-1]
ax.plot(x_line, y_line, 'r-', linewidth=2, label=f'Fitted line: y={m_hist[-1]:.2f}x+{b_hist[-1]:.2f}')
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_title('Final fitted line from gradient descent')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Summary

We've built up the key ideas for single-variable regression:

1. **The line:** A function y = mx + b with parameters we want to optimize
2. **The cost function:** Measures how badly our line fits the data (Mean Squared Error)
3. **Derivatives:** Tell us the slope of the cost function—which direction to move our parameters
4. **Gradient descent:** An iterative algorithm that uses derivatives to step downhill toward the optimal parameters

This same pattern—cost function + derivatives + gradient descent—is how neural networks learn too. In that case, the functions are more complex (many parameters, nonlinear), but the core idea is identical.

In [ ]:
# Exercise: Try different learning rates
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

learning_rates = [0.001, 0.01, 0.1]

for ax, lr in zip(axes, learning_rates):
    m_hist, b_hist, cost_hist = gradient_descent(x_data, y_data, m_init, b_init, lr, num_iterations)
    
    ax.plot(range(len(cost_hist)), cost_hist, 'b-', linewidth=2)
    ax.set_xlabel('Iteration')
    ax.set_ylabel('Cost')
    ax.set_title(f'Learning rate α = {lr}')
    ax.grid(True, alpha=0.3)
    ax.text(0.5, 0.95, f'Final cost: {cost_hist[-1]:.2f}', 
            transform=ax.transAxes, ha='center', va='top', 
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.show()

print("\nObservation: Learning rate matters!")
print("- Too small (0.001): slow progress")
print("- Just right (0.01): steady convergence")
print("- Too large (0.1): might oscillate or diverge")